# Aliasing demo

This notebook demonstrates the aliasing capabilities of the ACCESS-NRI Intake catalog.
Aliasing lets you search the catalog using familiar CMIP-style vocabulary (field names,
variable codes, frequency strings, etc.) and have those terms automatically mapped to
the underlying canonical values stored in the catalog.

For a full reference, see the [Aliasing](aliases.rst) documentation page.

> **Note**: This notebook must be run on Gadi with access to the `xp65` project
> and the relevant data projects. See the [How do I use it?](how.rst) page for prerequisites.

## Contents

1. [Setup](#setup)
2. [Field aliasing — CMIP column names](#field-aliasing)
3. [Value aliasing — frequency](#value-aliasing-frequency)
4. [Value aliasing — realm](#value-aliasing-realm)
5. [Value aliasing — model shortcuts](#value-aliasing-model)
6. [CMIP variable search (scalar)](#cmip-variable-scalar)
7. [CMIP variable search (list)](#cmip-variable-list)
8. [Regex passthrough](#regex-passthrough)
9. [Controlling alias warnings](#warnings)
10. [Escaping aliasing with .unwrap()](#unwrap)

<a name="setup"></a>
## 1. Setup

Load the catalog. On Gadi this uses the default catalog location in `/g/data/xp65`.

In [1]:
import warnings
import intake

cat = intake.cat.access_nri
cat

/home/189/ct1163/access-nri-intake-catalog/bin/venv/lib/python3.11/site-packages/fastprogress/fastprogress.py:107: UserWarning: Couldn't import ipywidgets properly, progress bar will use console behavior
  warn("Couldn't import ipywidgets properly, progress bar will use console behavior")


,model,description,realm,frequency,variable
name,,,,,
01deg_jra55_ryf_Control,{ACCESS-OM2-01},"{0.1° ACCESS-OM2 repeat year forcing control run for the simulations performed in Huguenin et al. (2024, GRL)}","{ocean, seaIce}","{fx, 1mon}","{total_ocean_swflx_vis, temp_yflux_adv, geolon_t, time_bnds, net_sfc_heating, opening_m, sw_heat, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, time, uocn_m, neutral, str..."
01deg_jra55_ryf_ENFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 El Níño run for the simulations performed in Huguenin et al. (2024, GRL)}","{ocean, seaIce}","{fx, 1mon}","{total_ocean_swflx_vis, geolon_t, time_bnds, net_sfc_heating, opening_m, sw_heat, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, neutral, strairx_m, fsalt_ai..."
01deg_jra55_ryf_LNFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 La Níña run for the simulations performed in Huguenin et al. (2024, GRL)}","{ocean, seaIce}","{fx, 1mon}","{total_ocean_swflx_vis, geolon_t, time_bnds, net_sfc_heating, opening_m, sw_heat, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, neutral, fsalt_ai_m, strairx..."
01deg_jra55v13_ryf9091,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991)},"{ocean, seaIce}","{1mon, 3mon, 1day, 3hr, fx}","{total_ocean_swflx_vis, geolon_t, temp_yflux_adv, net_sfc_heating, opening_m, sw_heat, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, xt_ocean_sub02, neutral..."
01deg_jra55v13_ryf9091_easterlies_down10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica decreased by 10%.},"{ocean, seaIce}","{fx, 1mon, 1day}","{total_ocean_swflx_vis, geolon_t, temp_yflux_adv, opening_m, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, time, uocn_m, neutral, fsalt_ai_m, strairx_m, alvdr_ai_m, total..."
01deg_jra55v13_ryf9091_easterlies_up10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica increased by 10%.},"{ocean, seaIce}","{fx, 1mon, 1day}","{total_ocean_swflx_vis, geolon_t, temp_yflux_adv, opening_m, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, time, uocn_m, neutral, strairx_m, fsalt_ai_m, total_ocean_runof..."
01deg_jra55v13_ryf9091_easterlies_up10_meridional,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and meridional wind speed around Antarctica increased by 10%.},"{ocean, seaIce}","{fx, 1mon, 1day}","{total_ocean_swflx_vis, temp_yflux_adv, geolon_t, opening_m, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, time, uocn_m, neutral, strairx_m, fsalt_ai_m, alvdr_ai_m, total..."
01deg_jra55v13_ryf9091_easterlies_up10_zonal,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal wind speed around Antarctica increased by 10%.},"{ocean, seaIce}","{fx, 1mon, 1day}","{total_ocean_swflx_vis, temp_yflux_adv, geolon_t, opening_m, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, time, uocn_m, neutral, fsalt_ai_m, strairx_m, alvdr_ai_m, total..."
01deg_jra55v13_ryf9091_qian_wthmp,{ACCESS-OM2},"{Future perturbations with wind, thermal and meltwater forcing, branching off 01deg_jra55v13_ryf9091, as described in Li et al. 2023, https://www.nature.com/articles/s41586-023-05762-w}","{ocean, seaIce}","{fx, 1mon}","{total_ocean_swflx_vis, geolon_t, opening_m, sw_heat, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, fsalt_ai_m, strairx_m, total_ocean_runoff, alvdr_ai_m, age_global, ty_tran..."


<a name="field-aliasing"></a>
## 2. Field aliasing — CMIP Style Variable ID's

The top-level catalog accepts CMIP-style search keywords in `.search()`. 

Each call will emit a `UserWarning` showing the mapping that was applied.

In [2]:
results = cat.search(variable="tas")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/494749647.py:2: UserWarning: Value aliasing: variable='tas' → variable=['fld_s03i236','tas']
  results = cat.search(variable="tas")


,model,description,realm,frequency,variable
name,,,,,
AUS2200,{AUS2200},"{""AUS2200 AMIP simulations collection. Each simulation is a limited area model study of the entire Australian continent at 2.2 km resolution, using the UM atmospheric model. The simulations cover ...",{atmos},"{1hr, subhr}",{tas}
HI_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP}",{atmos},"{3hr, 1mon, 1day}",{fld_s03i236}
HI_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP}",{atmos},"{1mon, 1day}",{fld_s03i236}
HI_nl_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP, and land-use change disabled}",{atmos},"{1mon, 1day}",{fld_s03i236}
HI_noluc_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP, and land-use change disabled}",{atmos},"{3hr, 1mon, 1day}",{fld_s03i236}
PI_GWL_B2035,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2035 }",{atmos},"{1mon, 1day}",{fld_s03i236}
PI_GWL_B2040,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2040}",{atmos},"{1mon, 1day}",{fld_s03i236}
PI_GWL_B2045,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2045}",{atmos},"{1mon, 1day}",{fld_s03i236}
PI_GWL_B2050,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2050}",{atmos},"{1mon, 1day}",{fld_s03i236}


<a name="value-aliasing-frequency"></a>
## 3. Value aliasing — frequency

Human-readable frequency strings are mapped to the catalog's stored values
(e.g. `"daily"` → `"1day"`). The search expands to include **both** terms, so
any data already labelled `"daily"` is also included.

In [3]:
# 'daily' expands to ['1day', 'daily']
results = cat.search(frequency="daily")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/327739463.py:2: UserWarning: Value aliasing: frequency='daily' → frequency=['1day','daily']
  results = cat.search(frequency="daily")


,model,description,realm,frequency,variable
name,,,,,
01deg_jra55v13_ryf9091,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991)},{ocean},{1day},"{uhrho_et, usurf, sw_ocean, average_DT, sfc_hflux_coupler, st_ocean, time_bounds, temp, u, time, vsurf, sfc_hflux_pme, surface_temp, xu_ocean, sfc_hflux_from_runoff, yt_ocean, sw_edges_ocean, aver..."
01deg_jra55v13_ryf9091_easterlies_down10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica decreased by 10%.},{ocean},{1day},"{usurf, average_DT, sfc_hflux_coupler, time_bounds, time, vsurf, sfc_hflux_pme, xu_ocean, surface_temp, sfc_hflux_from_runoff, yt_ocean, average_T1, tau_x, average_T2, surface_salt, mld, yu_ocean,..."
01deg_jra55v13_ryf9091_easterlies_up10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica increased by 10%.},{ocean},{1day},"{usurf, average_DT, sfc_hflux_coupler, time_bounds, time, vsurf, sfc_hflux_pme, xu_ocean, surface_temp, sfc_hflux_from_runoff, yt_ocean, average_T1, tau_x, average_T2, surface_salt, mld, yu_ocean,..."
01deg_jra55v13_ryf9091_easterlies_up10_meridional,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and meridional wind speed around Antarctica increased by 10%.},{ocean},{1day},"{usurf, average_DT, sfc_hflux_coupler, time_bounds, time, vsurf, sfc_hflux_pme, xu_ocean, surface_temp, sfc_hflux_from_runoff, yt_ocean, average_T1, tau_x, average_T2, surface_salt, mld, yu_ocean,..."
01deg_jra55v13_ryf9091_easterlies_up10_zonal,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal wind speed around Antarctica increased by 10%.},{ocean},{1day},"{usurf, average_DT, sfc_hflux_coupler, time_bounds, time, vsurf, sfc_hflux_pme, surface_temp, xu_ocean, sfc_hflux_from_runoff, yt_ocean, average_T1, average_T2, tau_x, surface_salt, mld, yu_ocean,..."
01deg_jra55v13_ryf9091_weddell_down2,{ACCESS-OM2-01},"{Weddell Sea decreased meltwater perturbation experiment, branched off 01deg_jra55v13_ryf9091. }",{ocean},{1day},"{usurf, average_DT, sfc_hflux_coupler, time_bounds, time, vsurf, sfc_hflux_pme, surface_temp, xu_ocean, sfc_hflux_from_runoff, yt_ocean, average_T1, average_T2, tau_x, surface_salt, mld, yu_ocean,..."
01deg_jra55v13_ryf9091_weddell_up1,{ACCESS-OM2-01},"{Weddell Sea increased meltwater perturbation experiment, branched off 01deg_jra55v13_ryf9091. }",{ocean},{1day},"{usurf, average_DT, sfc_hflux_coupler, time_bounds, time, vsurf, sfc_hflux_pme, xu_ocean, surface_temp, sfc_hflux_from_runoff, yt_ocean, average_T1, tau_x, average_T2, surface_salt, mld, yu_ocean,..."
01deg_jra55v140_iaf,{ACCESS-OM2-01},{Cycle 1 of 0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.4.0 OMIP2 interannual forcing},"{ocean, seaIce}",{1day},"{total_ocean_swflx_vis, time_bounds, total_ocean_calving_heat, time, snoice, total_ocean_runoff, scalar_axis, blkmask, average_T2, surface_salt, mld, aice, wt, pe_tot, nv, pme_river, total_ocean_c..."
01deg_jra55v140_iaf_cycle2,{ACCESS-OM2-01},{Cycle 2 of 0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.4.0 OMIP2 interannual forcing},"{ocean, seaIce}",{1day},"{total_ocean_swflx_vis, time_bounds, time, snoice, total_ocean_runoff, scalar_axis, surface_temp_min, blkmask, average_T2, meltb, surface_salt, mld, aice, pe_tot, nv, fcondtop_ai, pme_river, surfa..."


In [4]:
# 'monthly' expands to ['1mon', 'monthly']
results = cat.search(frequency="monthly")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/4282274679.py:2: UserWarning: Value aliasing: frequency='monthly' → frequency=['1mon','monthly']
  results = cat.search(frequency="monthly")


,model,description,realm,frequency,variable
name,,,,,
01deg_jra55_ryf_Control,{ACCESS-OM2-01},"{0.1° ACCESS-OM2 repeat year forcing control run for the simulations performed in Huguenin et al. (2024, GRL)}","{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, temp_yflux_adv, time_bnds, net_sfc_heating, opening_m, ty_trans_submeso, sw_heat, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, neutral, fsalt_ai_m, s..."
01deg_jra55_ryf_ENFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 El Níño run for the simulations performed in Huguenin et al. (2024, GRL)}","{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, time_bnds, net_sfc_heating, opening_m, sw_heat, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, neutral, fsalt_ai_m, strairx_m, alvdr_..."
01deg_jra55_ryf_LNFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 La Níña run for the simulations performed in Huguenin et al. (2024, GRL)}","{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, time_bnds, net_sfc_heating, opening_m, ty_trans_submeso, sw_heat, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, neutral, fsalt_ai_m, strairx_m, alvdr_..."
01deg_jra55v13_ryf9091,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991)},"{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, temp_yflux_adv, opening_m, net_sfc_heating, ty_trans_submeso, sw_heat, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, neutral, strairx_m, fsalt_ai_m, t..."
01deg_jra55v13_ryf9091_easterlies_down10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica decreased by 10%.},"{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, temp_yflux_adv, opening_m, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, time, uocn_m, neutral, strairx_m, fsalt_ai_m, alvdr_ai_m, total_ocean_run..."
01deg_jra55v13_ryf9091_easterlies_up10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica increased by 10%.},"{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, temp_yflux_adv, opening_m, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, neutral, strairx_m, fsalt_ai_m, alvdr_ai_m, total_ocean_run..."
01deg_jra55v13_ryf9091_easterlies_up10_meridional,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and meridional wind speed around Antarctica increased by 10%.},"{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, temp_yflux_adv, opening_m, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, time, uocn_m, neutral, strairx_m, fsalt_ai_m, alvdr_ai_m, total_ocean_run..."
01deg_jra55v13_ryf9091_easterlies_up10_zonal,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal wind speed around Antarctica increased by 10%.},"{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, temp_yflux_adv, opening_m, ty_trans_submeso, time_bounds, total_ocean_calving_heat, swflx, divu_m, uocn_m, time, neutral, strairx_m, fsalt_ai_m, total_ocean_runoff, alvdr_a..."
01deg_jra55v13_ryf9091_qian_wthmp,{ACCESS-OM2},"{Future perturbations with wind, thermal and meltwater forcing, branching off 01deg_jra55v13_ryf9091, as described in Li et al. 2023, https://www.nature.com/articles/s41586-023-05762-w}","{ocean, seaIce}",{1mon},"{total_ocean_swflx_vis, opening_m, sw_heat, time_bounds, total_ocean_calving_heat, swflx, divu_m, time, uocn_m, fsalt_ai_m, strairx_m, alvdr_ai_m, total_ocean_runoff, age_global, ty_trans_int_z, s..."


<a name="value-aliasing-realm"></a>
## 4. Value aliasing — realm

Common realm strings map to the catalog's stored values
(e.g. `"atmosphere"` → `"atmos"`, `"sea-ice"` → `"seaIce"`).

In [5]:
# 'atmosphere' expands to ['atmos', 'atmosphere']
results = cat.search(realm="atmosphere")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/1345224546.py:2: UserWarning: Value aliasing: realm='atmosphere' → realm=['atmos','atmosphere']
  results = cat.search(realm="atmosphere")


,model,description,realm,frequency,variable
name,,,,,
AUS2200,{AUS2200},"{""AUS2200 AMIP simulations collection. Each simulation is a limited area model study of the entire Australian continent at 2.2 km resolution, using the UM atmospheric model. The simulations cover ...",{atmos},"{3hr, 6hr, 1hr, subhr}","{clmxro, ua, pralsprof, rainmxrat, cllow, clw, clmed, eow, clhigh, hfss, rsdsdiff, reflmax, rsut, evspsbl, psl, rlds, z0, va, cl, rls, wsgmax10m, zg, rsds, pfull, huss, grplmxrat, vas, rss, theta,..."
HI_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP}",{atmos},"{3hr, 6hr, 1mon, 1day}","{fld_s03i897, fld_s30i208, fld_s16i222, fld_s03i316, fld_s08i223, fld_s03i298, fld_s00i103, fld_s03i460, fld_s03i245_max, fld_s03i916, fld_s03i395, fld_s08i231, fld_s00i010, fld_s03i904, fld_s03i3..."
HI_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP}",{atmos},"{1mon, 1day}","{fld_s03i897, fld_s30i208, fld_s03i316, fld_s16i222, fld_s08i223, fld_s03i298, fld_s00i103, fld_s03i460, fld_s03i916, fld_s02i206, fld_s03i245_max, fld_s03i395, fld_s08i234, fld_s03i904, fld_s00i1..."
HI_nl_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP, and land-use change disabled}",{atmos},"{1mon, 1day}","{fld_s03i897, fld_s30i208, fld_s03i316, fld_s16i222, fld_s08i223, fld_s03i811, fld_s03i298, fld_s00i103, fld_s08i231, fld_s03i245_max, fld_s03i916, fld_s03i460, fld_s08i234, fld_s00i010, fld_s02i2..."
HI_noluc_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP, and land-use change disabled}",{atmos},"{3hr, 6hr, 1mon, 1day}","{fld_s03i897, fld_s30i208, fld_s16i222, fld_s03i316, fld_s08i223, fld_s03i298, fld_s00i103, fld_s03i460, fld_s02i206, fld_s08i231, fld_s03i245_max, fld_s08i234, fld_s00i010, fld_s03i904, fld_s03i3..."
PI_GWL_B2035,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2035 }",{atmos},"{1mon, 1day}","{fld_s30i208, fld_s16i222, fld_s08i223, fld_s03i298, fld_s00i103, fld_s08i231, fld_s03i395, fld_s03i245_max, fld_s02i206, fld_s00i116, fld_s00i010, fld_s03i460, fld_s05i251, fld_s08i234, fld_s03i3..."
PI_GWL_B2040,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2040}",{atmos},"{1mon, 1day}","{fld_s30i208, fld_s16i222, fld_s08i223, fld_s03i298, fld_s00i103, fld_s03i811, fld_s03i245_max, fld_s03i460, fld_s08i231, fld_s08i234, fld_s00i010, fld_s02i206, fld_s05i251, fld_s03i331, fld_s03i3..."
PI_GWL_B2045,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2045}",{atmos},"{1mon, 1day}","{fld_s30i208, fld_s16i222, fld_s08i223, fld_s03i298, fld_s00i103, fld_s03i460, fld_s02i206, fld_s03i811, fld_s03i245_max, fld_s08i234, fld_s00i116, fld_s00i010, fld_s05i251, fld_s03i331, fld_s03i3..."
PI_GWL_B2050,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2050}",{atmos},"{1mon, 1day}","{fld_s30i208, fld_s16i222, fld_s08i223, fld_s03i298, fld_s00i103, fld_s03i395, fld_s02i206, fld_s03i460, fld_s03i245_max, fld_s08i234, fld_s00i010, fld_s08i231, fld_s05i251, fld_s00i116, fld_s03i3..."


In [6]:
# 'sea-ice' expands to ['seaIce', 'sea-ice']
results = cat.search(realm="sea-ice")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/1193532718.py:2: UserWarning: Value aliasing: realm='sea-ice' → realm=['seaIce','sea-ice']
  results = cat.search(realm="sea-ice")


,model,description,realm,frequency,variable
name,,,,,
01deg_jra55_ryf_Control,{ACCESS-OM2-01},"{0.1° ACCESS-OM2 repeat year forcing control run for the simulations performed in Huguenin et al. (2024, GRL)}",{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, uarea, fmeltt_ai_m, alvdf_ai_m, time_bounds, vocn_m, divu_m, uocn_m, mlt_onset_m, time, hi_m, vvel_m, uvel_m, aice_m, strairx_m, ..."
01deg_jra55_ryf_ENFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 El Níño run for the simulations performed in Huguenin et al. (2024, GRL)}",{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, uarea, fmeltt_ai_m, alvdf_ai_m, time_bounds, vocn_m, divu_m, uocn_m, time, mlt_onset_m, hi_m, vvel_m, uvel_m, aice_m, ULON, strai..."
01deg_jra55_ryf_LNFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 La Níña run for the simulations performed in Huguenin et al. (2024, GRL)}",{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, fmeltt_ai_m, uarea, alvdf_ai_m, time_bounds, vocn_m, divu_m, uocn_m, time, mlt_onset_m, hi_m, vvel_m, aice_m, uvel_m, ULON, strai..."
01deg_jra55v13_ryf9091,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991)},{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, uarea, fmeltt_ai_m, time_bounds, alvdf_ai_m, vocn_m, divu_m, time, mlt_onset_m, uocn_m, hi_m, vvel_m, aice_m, uvel_m, ULON, fsalt..."
01deg_jra55v13_ryf9091_easterlies_down10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica decreased by 10%.},{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, fmeltt_ai_m, uarea, alvdf_ai_m, time_bounds, vocn_m, divu_m, uocn_m, mlt_onset_m, time, hi_m, vvel_m, aice_m, uvel_m, ULON, fsalt..."
01deg_jra55v13_ryf9091_easterlies_up10,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal/meridional wind speed around Antarctica increased by 10%.},{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, uarea, fmeltt_ai_m, alvdf_ai_m, time_bounds, vocn_m, divu_m, uocn_m, time, mlt_onset_m, hi_m, vvel_m, uvel_m, aice_m, ULON, fsalt..."
01deg_jra55v13_ryf9091_easterlies_up10_meridional,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and meridional wind speed around Antarctica increased by 10%.},{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, fmeltt_ai_m, uarea, time_bounds, alvdf_ai_m, vocn_m, divu_m, uocn_m, mlt_onset_m, time, hi_m, vvel_m, aice_m, uvel_m, fsalt_ai_m,..."
01deg_jra55v13_ryf9091_easterlies_up10_zonal,{ACCESS-OM2-01},{0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.3 RYF9091 repeat year forcing (May 1990 to Apr 1991) and zonal wind speed around Antarctica increased by 10%.},{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, uarea, fmeltt_ai_m, time_bounds, alvdf_ai_m, vocn_m, divu_m, time, mlt_onset_m, uocn_m, hi_m, vvel_m, uvel_m, aice_m, ULON, strai..."
01deg_jra55v13_ryf9091_qian_wthmp,{ACCESS-OM2},"{Future perturbations with wind, thermal and meltwater forcing, branching off 01deg_jra55v13_ryf9091, as described in Li et al. 2023, https://www.nature.com/articles/s41586-023-05762-w}",{seaIce},{1mon},"{tarea, dyu, alidf_ai_m, ANGLE, dxu, opening_m, ANGLET, strairy_m, fmeltt_ai_m, uarea, time_bounds, alvdf_ai_m, vocn_m, divu_m, uocn_m, time, mlt_onset_m, hi_m, vvel_m, uvel_m, aice_m, ULON, fsalt..."


<a name="value-aliasing-model"></a>
## 5. Value aliasing — model shortcuts

Short model names map to their full catalog names.

In [7]:
# 'ACCESS-ESM1' expands to ['ACCESS-ESM1-5', 'ACCESS-ESM1']
results = cat.search(model="ACCESS-ESM1")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/2543793196.py:2: UserWarning: Value aliasing: model='ACCESS-ESM1' → model=['ACCESS-ESM1-5','ACCESS-ESM1']
  results = cat.search(model="ACCESS-ESM1")


,model,description,realm,frequency,variable
name,,,,,
HI_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP}","{atmos, ocean, seaIce}","{1yr, 1mon, 1day, 3hr, 6hr}","{wind_power_v, fld_s30i208, geolon_t, dic, fld_s03i298, fmeltt_ai, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."
HI_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP}","{atmos, ocean, seaIce}","{1yr, 1mon, 1day}","{wind_power_v, fld_s30i208, geolon_t, dic, fmeltt_ai, fld_s03i298, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."
HI_nl_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP, and land-use change disabled}","{atmos, ocean, seaIce}","{1yr, 1mon, 1day}","{wind_power_v, fld_s30i208, geolon_t, dic, fmeltt_ai, fld_s03i298, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."
HI_noluc_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP, and land-use change disabled}","{atmos, ocean, seaIce}","{1yr, 1mon, 1day, 3hr, 6hr}","{wind_power_v, fld_s30i208, dic, geolon_t, fmeltt_ai, fld_s03i298, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."
PI_GWL_B2035,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2035 }","{atmos, ocean, seaIce}","{1yr, 1mon, 1day}","{wind_power_v, fld_s30i208, geolon_t, dic, fmeltt_ai, fld_s03i298, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."
PI_GWL_B2040,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2040}","{atmos, ocean, seaIce}","{1yr, 1mon, 1day}","{wind_power_v, fld_s30i208, geolon_t, dic, fmeltt_ai, fld_s03i298, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."
PI_GWL_B2045,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2045}","{atmos, ocean, seaIce}","{1yr, 1mon, 1day}","{wind_power_v, fld_s30i208, dic, geolon_t, fld_s03i298, fmeltt_ai, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."
PI_GWL_B2050,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2050}","{atmos, ocean, seaIce}","{1yr, 1mon, 1day}","{wind_power_v, fld_s30i208, geolon_t, dic, fmeltt_ai, fld_s03i298, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."
PI_GWL_B2055,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2055}","{atmos, ocean, seaIce}","{1yr, 1mon, 1day}","{wind_power_v, fld_s30i208, geolon_t, dic, fmeltt_ai, fld_s03i298, fld_s02i206, rossby, dht, urhod, temp_yflux_ndiffuse_int_z, fld_s04i203, fld_s03i100, time, salt_yflux_adv, fld_s01i235, fld_s30i..."


<a name="cmip-variable-scalar"></a>
## 6. CMIP variable search — scalar

CMIP standard variable names (e.g. `tas`, `pr`, `ci`) are mapped to the raw
ACCESS field codes stored in ESM datastores. The search always includes *both*
the CMIP name and the ACCESS code, so you never miss data labelled either way.

These mappings come from `access-esm1-6-cmip-mappings.json` which covers 137
atmosphere, land, and ocean variables for ACCESS-ESM1.5 and ACCESS-ESM1.6.

In [8]:
ds = cat['HI_CN_05']
ds

,unique
filename,9435
path,9435
file_id,16
frequency,5
start_date,3705
end_date,3705
variable,655
variable_long_name,551
variable_standard_name,138
variable_cell_methods,8


In [9]:
# Search for 'tas' — CMIP name for near-surface air temperature
# Automatically expands to variable=['fld_s03i236', 'tas']
results = ds.search(variable="tas")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/4195522969.py:3: UserWarning: Value aliasing: variable='tas' → variable=['fld_s03i236','tas']
  results = ds.search(variable="tas")


,unique
filename,4740
path,4740
file_id,7
frequency,3
start_date,2760
end_date,2760
variable,276
variable_long_name,202
variable_standard_name,90
variable_cell_methods,5


In [10]:
# 'ci' (convection time fraction) → ['fld_s05i269', 'ci']
results = ds.search(variable="ci")
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/3266990395.py:2: UserWarning: Value aliasing: variable='ci' → variable=['fld_s05i269','ci']
  results = ds.search(variable="ci")


,unique
filename,0
path,0
file_id,0
frequency,0
start_date,0
end_date,0
variable,0
variable_long_name,0
variable_standard_name,0
variable_cell_methods,0


<a name="cmip-variable-list"></a>
## 7. CMIP variable search — list

You can pass a list of CMIP variable names. Each is aliased individually and
the full expanded set is searched.

In [11]:
# Each CMIP name is resolved: ci→fld_s05i269, cl→fld_s02i261, tas→fld_s03i236
results = ds.search(variable=["ci", "cl", "tas"])
results

/jobfs/162823973.gadi-pbs/ipykernel_3198941/1139839834.py:2: UserWarning: Value aliasing: variable='['ci', 'cl', 'tas']' → variable=['fld_s02i261', 'tas', 'fld_s03i236', 'cl', 'ci', 'fld_s05i269']
  results = ds.search(variable=["ci", "cl", "tas"])


,unique
filename,4740
path,4740
file_id,7
frequency,3
start_date,2760
end_date,2760
variable,276
variable_long_name,202
variable_standard_name,90
variable_cell_methods,5


<a name="regex-passthrough"></a>
## 8. Regex passthrough

Regex patterns and other non-string values bypass aliasing entirely and are
passed directly to the underlying catalog. This allows you to use the full power
of intake-esm's regex search when you need it.

In [12]:
# Regex string — passed through unchanged, no aliasing applied - so we don't expect ay results
results = ds.search(variable="ci|cl|tas")
results

,unique
filename,0
path,0
file_id,0
frequency,0
start_date,0
end_date,0
variable,0
variable_long_name,0
variable_standard_name,0
variable_cell_methods,0


<a name="warnings"></a>
## 9. Controlling alias warnings

Each alias expansion emits a `UserWarning`. You can suppress these using
Python's standard `warnings` module.

In [13]:
# Suppress UserWarnings from aliasing
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    results = cat.search(frequency="daily", realm="atmosphere")

results

,model,description,realm,frequency,variable
name,,,,,
HI_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP}",{atmos},{1day},"{fld_s30i208, fld_s16i222, fld_s03i810, fld_s03i236_max, fld_s03i209, pseudo_level, fld_s08i223, fld_s03i316, fld_s05i205, fld_s30i204, fld_s03i807, fld_s02i206, fld_s03i245_max, fld_s08i234, fld_..."
HI_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP}",{atmos},{1day},"{fld_s30i208, fld_s03i810, fld_s03i236_max, fld_s08i223, fld_s16i222, fld_s03i316, pseudo_level, fld_s03i209, fld_s05i205, fld_s30i204, fld_s02i206, fld_s03i807, fld_s03i245_max, fld_s08i234, fld_..."
HI_nl_C_05_r1,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with nitrogen and phosphorus limitations disabled within CASA-CNP, and land-use change disabled}",{atmos},{1day},"{fld_s30i208, fld_s03i810, fld_s03i316, fld_s16i222, fld_s03i209, fld_s03i236_max, fld_s08i223, pseudo_level, fld_s30i204, fld_s05i205, fld_s02i206, fld_s03i807, fld_s03i245_max, fld_s08i234, fld_..."
HI_noluc_CN_05,{ACCESS-ESM1-5},"{Historical run using same configuration as CMIP6 ACCESS-ESM1.5 historical r1i1p1f1, but with phosphorus limitation disabled within CASA-CNP, and land-use change disabled}",{atmos},{1day},"{fld_s30i208, fld_s03i316, fld_s16i222, fld_s08i223, fld_s03i209, pseudo_level, fld_s03i810, fld_s03i236_max, fld_s30i204, fld_s05i205, fld_s03i245_max, fld_s02i206, fld_s03i807, fld_s08i234, fld_..."
PI_GWL_B2035,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2035 }",{atmos},{1day},"{fld_s03i810, fld_s03i236_max, fld_s08i223, fld_s16i222, pseudo_level, fld_s02i206, fld_s03i245_max, fld_s03i807, fld_s08i230, fld_s03i808, fld_s05i216, fld_s03i245_min, fld_s03i236_min, fld_s00i0..."
PI_GWL_B2040,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2040}",{atmos},{1day},"{fld_s03i236_max, fld_s16i222, fld_s08i223, fld_s03i810, pseudo_level, fld_s03i245_max, fld_s03i807, fld_s02i206, fld_s08i230, fld_s03i808, fld_s05i216, fld_s03i245_min, fld_s03i236_min, fld_s03i2..."
PI_GWL_B2045,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2045}",{atmos},{1day},"{fld_s03i810, fld_s03i236_max, fld_s08i223, fld_s16i222, pseudo_level, fld_s02i206, fld_s03i245_max, fld_s03i807, fld_s08i230, fld_s03i808, fld_s05i216, fld_s03i245_min, fld_s03i236_min, fld_s03i2..."
PI_GWL_B2050,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2050}",{atmos},{1day},"{fld_s16i222, fld_s03i810, fld_s08i223, fld_s03i236_max, pseudo_level, fld_s03i807, fld_s03i245_max, fld_s02i206, fld_s08i230, fld_s03i808, fld_s05i216, fld_s03i245_min, fld_s03i236_min, fld_s00i0..."
PI_GWL_B2055,{ACCESS-ESM1-5},"{Climate stabilization run at different global warming levels with zero C02 emissions and pre-industrial aerosols, starting in 2055}",{atmos},{1day},"{fld_s03i810, fld_s03i236_max, fld_s08i223, fld_s16i222, pseudo_level, fld_s03i807, fld_s03i245_max, fld_s02i206, fld_s08i230, fld_s03i808, fld_s05i216, fld_s03i245_min, fld_s03i236_min, fld_s00i0..."


<a name="unwrap"></a>
## 10. Escaping aliasing with `.unwrap()`

Both the top-level catalog wrapper and the per-dataset wrapper expose an
`.unwrap()` method that returns the raw underlying catalog object with no
aliasing layer. Use this when you need the exact type expected by another
library or want to ensure no aliasing occurs

In [14]:
# Get the raw DfFileCatalog (no aliasing wrapper)
raw_cat = cat.unwrap()
type(raw_cat)

intake_dataframe_catalog.core.DfFileCatalog

In [15]:
# Get the raw esm_datastore (no aliasing wrapper)
raw_ds = ds.unwrap()
type(raw_ds)

intake_esm.core.esm_datastore